<a href="https://colab.research.google.com/github/visal1411/INTERNSHIP/blob/ML/Predict_onecow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Score a single new cow using the already-trained IsolationForest.
===================================================================
Run train_isolation_forest.py FIRST — it produces
isolation_forest_artifacts.joblib, which this script loads.

Usage:
    python3 predict_single_cow.py

Or import and call score_cow() directly from a notebook cell, e.g.
Colab:

    from predict_single_cow import score_cow
    score_cow(breed="Local Zebu", age=14, gender="Male", weight=95.0)
"""

import numpy as np
import pandas as pd
import joblib

ARTIFACTS_PATH = "isolation_forest_artifacts.joblib"


def load_artifacts(path=ARTIFACTS_PATH):
    return joblib.load(path)


def score_cow(breed: str, age: int, gender: str, weight: float,
              artifacts=None, verbose: bool = True):
    """
    Score one new cow against the already-trained model.

    Parameters
    ----------
    breed, gender : must match a category the model was trained on
                     (e.g. "Local Zebu", "Male") - an unseen breed/gender
                     will raise an error, since the model has no basis
                     to judge it.
    age            : in months
    weight         : in kg

    Returns a dict with the anomaly score and flag.
    """
    art = artifacts or load_artifacts()
    cohort_stats = art["cohort_stats"]
    le_breed, le_gender = art["le_breed"], art["le_gender"]
    model = art["model"]

    # 1. bucket age the same way training data was bucketed
    age_group = pd.cut(
        [age], bins=art["age_bins"], labels=art["age_labels"], right=True
    )[0]

    # 2. look up this cow's cohort (Breed + Gender + AgeGroup) stats
    match = cohort_stats[
        (cohort_stats["Breed"] == breed)
        & (cohort_stats["Gender"] == gender)
        & (cohort_stats["AgeGroup"] == age_group)
    ]

    if match.empty:
        # No cohort seen in training with this exact combo -> can't compute
        # a reliable z-score. Fall back to the closest available cohort
        # (same breed+gender, nearest age group) rather than guessing.
        fallback = cohort_stats[
            (cohort_stats["Breed"] == breed) & (cohort_stats["Gender"] == gender)
        ]
        if fallback.empty:
            raise ValueError(
                f"No training data at all for Breed='{breed}', Gender='{gender}'. "
                "Can't score this cow reliably — expand the training set."
            )
        cohort_mean = fallback["CohortMeanWeight"].mean()
        cohort_std = fallback["CohortStdWeight"].mean()
        if verbose:
            print(f"[warning] No exact cohort for age group '{age_group}'. "
                  f"Using average across all age groups for this breed/gender instead.")
    else:
        cohort_mean = match["CohortMeanWeight"].values[0]
        cohort_std = match["CohortStdWeight"].values[0]

    weight_zscore = (weight - cohort_mean) / cohort_std

    # 3. encode + build feature row in the SAME column order used in training
    try:
        breed_enc = le_breed.transform([breed])[0]
        gender_enc = le_gender.transform([gender])[0]
    except ValueError as e:
        raise ValueError(
            f"Unseen category: {e}. This breed/gender wasn't in the training data."
        )

    feature_row = pd.DataFrame([{
        "Breed_enc": breed_enc,
        "Gender_enc": gender_enc,
        "Age": age,
        "Weight_Zscore": weight_zscore,
    }])[art["feature_names"]]

    anomaly_score = model.decision_function(feature_row)[0]
    is_anomaly = model.predict(feature_row)[0] == -1

    if not is_anomaly:
        flag = "Normal"
    elif weight_zscore < 0:
        flag = "Flagged - Potential Sickness"
    elif gender == "Female":
        flag = "Flagged - Possible Pregnancy or Overweight"
    else:
        flag = "Flagged - Unusual, Needs Review"

    result = {
        "Breed": breed,
        "Age": age,
        "AgeGroup": str(age_group),
        "Gender": gender,
        "Weight": weight,
        "CohortMeanWeight": round(cohort_mean, 1),
        "Weight_Zscore": round(weight_zscore, 3),
        "AnomalyScore": round(anomaly_score, 4),
        "Flag": flag,
    }

    if verbose:
        print("\n--- Cow scoring result ---")
        for k, v in result.items():
            print(f"{k:>18}: {v}")

    return result


if __name__ == "__main__":
    # ---- Example test cases — edit these or replace with real input ----
    print("=== Test 1: a plausible healthy cow ===")
    score_cow(breed="Sahiwal", age=24, gender="Female", weight=200.0)

    print("\n=== Test 2: a clearly underweight cow (should flag Potential Sickness) ===")
    score_cow(breed="Sahiwal", age=24, gender="Female", weight=90.0)

    print("\n=== Test 3: a clearly overweight female (should flag Pregnancy/Overweight) ===")
    score_cow(breed="Sahiwal", age=24, gender="Female", weight=340.0)

    print("\n=== Test 4: a clearly overweight male (should flag Unusual/Needs Review) ===")
    score_cow(breed="Sahiwal", age=24, gender="Male", weight=380.0)

=== Test 1: a plausible healthy cow ===


FileNotFoundError: [Errno 2] No such file or directory: 'isolation_forest_artifacts.joblib'